**Importamos librerias**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

**Configuración**

In [0]:
CATALOG = "proyecto_smart_claims"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark = SparkSession.builder.getOrCreate()

**Leemos las tablas bronze**

In [0]:
customers = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

policies = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.policies")

claims = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.claims")

telematics = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.telematics")

training_images = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.training_imgs")

claim_images = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images")

claim_images_metadata = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images_metadata")

**Trasnformaciones**

Customers

In [0]:
customers_clean = (
    customers
    .withColumn(
        "date_of_birth",
        F.coalesce(
            F.try_to_timestamp(F.col("date_of_birth"), F.lit("MM/dd/yyyy")),
            F.try_to_timestamp(F.col("date_of_birth"), F.lit("dd/MM/yyyy")),
            F.try_to_timestamp(F.col("date_of_birth"), F.lit("yyyy-MM-dd")),
            F.try_to_timestamp(F.col("date_of_birth"), F.lit("dd-MM-yyyy")),
            F.try_to_timestamp(F.col("date_of_birth"), F.lit("yyyy/MM/dd"))
        ).cast("date")
    )
    .withColumn("name", F.trim(F.col("name")))
    # Formato: 'Apellido, Nombre' → split por ', '
    .withColumn("last_name",  F.split(F.col("name"), ", ").getItem(0))
    .withColumn("first_name", F.split(F.col("name"), ", ").getItem(1))
    .withColumn(
        "address",
        F.concat_ws(", ",
            F.coalesce(F.col("borough"),      F.lit("")),
            F.coalesce(F.col("neighborhood"), F.lit("")),
            F.coalesce(F.col("zip_code"),     F.lit(""))
        )
    )
    .filter(F.col("date_of_birth").isNotNull())
    .filter(F.col("name").isNotNull() & (F.trim(F.col("name")) != ""))
    .filter(F.col("first_name").isNotNull() & (F.col("first_name") != ""))
    .filter(F.col("last_name").isNotNull() & (F.col("last_name") != ""))
)

Policies

In [0]:
policies_clean = (
    policies
    # Asegurar tipo correcto en premium
    .withColumn("PREMIUM", F.col("PREMIUM").cast("double"))

    # Castear fechas antes de comparar
    .withColumn("POL_EFF_DATE",    F.to_date(F.col("POL_EFF_DATE")))
    .withColumn("POL_EXPIRY_DATE", F.to_date(F.col("POL_EXPIRY_DATE")))

    # Validar campos clave
    .filter(F.col("POLICY_NO").isNotNull())
    .filter(F.col("CUST_ID").isNotNull())

    # Validación básica de negocio
    .filter(F.col("PREMIUM") >= 0)

    # Validación de fechas
    .filter(F.col("POL_EFF_DATE").isNotNull())
    .filter(F.col("POL_EXPIRY_DATE").isNotNull())
    .filter(F.col("POL_EFF_DATE") <= F.col("POL_EXPIRY_DATE"))
)

Claims

In [0]:
cols_to_drop = [c for c in ["date", "date_1", "date_2"] if c in claims.columns]

claims_clean = (
    claims
    # Validar claves
    .filter(F.col("claim_no").isNotNull())
    .filter(F.col("policy_no").isNotNull())

    # Convertir las tres fechas requeridas
    .withColumn("claim_date",        F.to_date(F.col("claim_date")))
    .withColumn("license_issue_date",F.to_date(F.col("license_issue_date")))

    # Validaciones básicas
    .filter(F.col("total") >= 0)
    .filter(F.col("age") > 0)

    # Drop seguro: solo elimina columnas que existen
    .drop(*cols_to_drop)
)

Telematics

In [0]:
telematics_clean = (
    telematics
    # Convertir timestamp
    .withColumn("event_timestamp", F.to_timestamp(F.col("event_timestamp")))

    # Validar clave
    .filter(F.col("chassis_no").isNotNull())

    # Filtrar coordenadas válidas (excluye nulos también)
    .filter(
        F.col("latitude").isNotNull() & F.col("longitude").isNotNull() &
        (F.col("latitude").between(-90, 90)) &
        (F.col("longitude").between(-180, 180))
    )

    # Validar velocidad
    .filter(F.col("speed") >= 0)
)

Training images

In [0]:
training_images_clean = (
    training_images
    # Filtrar paths nulos antes de procesar
    .filter(F.col("path").isNotNull())

    # Extraer nombre del archivo
    .withColumn("image_name", F.element_at(F.split(F.col("path"), "/"), -1))

    # Extraer label (antes del '_')
    .withColumn("label", F.split(F.col("image_name"), "_").getItem(0))

    # Filtrar labels nulos o vacíos
    .filter(F.col("label").isNotNull() & (F.col("label") != ""))
)

Claim images

In [0]:
claim_images_clean = (
    claim_images
    # Filtrar paths nulos antes de procesar
    .filter(F.col("path").isNotNull())

    # Extraer nombre del archivo
    .withColumn("image_name", F.element_at(F.split(F.col("path"), "/"), -1))

    # Filtrar image_name nulos
    .filter(F.col("image_name").isNotNull() & (F.col("image_name") != ""))
)

Claim images metadata

In [0]:
metadata_clean = (
    claim_images_metadata
    # Seleccionar columnas necesarias
    .select("image_name", "image_id", "claim_no", "chassis_no")

    # Limpiar nulos clave
    .filter(F.col("image_name").isNotNull())
    .filter(F.col("claim_no").isNotNull())

    # Eliminar duplicados
    .dropDuplicates(["image_name"])
)

**Guardamos las tablas**

In [0]:
customers_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean")

policies_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.policies_clean")

claims_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.claims_clean")

telematics_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.telematics_clean")

training_images_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.training_images")

claim_images_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.claim_images")

metadata_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.claim_images_metadata_clean")

In [0]:
display(spark.sql(f"""
SELECT 
    date_of_birth,
    name,
    first_name,
    last_name,
    address
FROM {CATALOG}.{SILVER_SCHEMA}.customers_clean
LIMIT 10
"""))

In [0]:
# ============================================================
# PUNTO 6: Validación de la capa Silver
# ============================================================

from pyspark.sql import functions as F

CATALOG = "proyecto_smart_claims"
SILVER_SCHEMA = "silver"

# ------------------------------------------------------------
# 1. CUSTOMERS CLEAN
# ------------------------------------------------------------
print("=" * 60)
print("CUSTOMERS CLEAN")
print("=" * 60)

customers_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean")

# Fechas no sean texto (tipo date)
print("\n>>> Tipo de dato de date_of_birth (debe ser 'date', no string):")
customers_silver.printSchema()

# Campos derivados existen y no tienen nulos
print("\n>>> Nulos en campos clave:")
customers_silver.select(
    F.count(F.when(F.col("date_of_birth").isNull(), 1)).alias("nulos_date_of_birth"),
    F.count(F.when(F.col("name").isNull() | (F.col("name") == ""), 1)).alias("nulos_name"),
    F.count(F.when(F.col("first_name").isNull() | (F.col("first_name") == ""), 1)).alias("nulos_first_name"),
    F.count(F.when(F.col("last_name").isNull() | (F.col("last_name") == ""), 1)).alias("nulos_last_name"),
    F.count(F.when(F.col("address").isNull() | (F.col("address") == ""), 1)).alias("nulos_address"),
).show()

# Muestra de los campos derivados
print("\n>>> Muestra de campos derivados:")
customers_silver.select("date_of_birth", "name", "first_name", "last_name", "address").show(5, truncate=False)

# ------------------------------------------------------------
# 2. POLICIES CLEAN
# ------------------------------------------------------------
print("=" * 60)
print("POLICIES CLEAN")
print("=" * 60)

policies_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.policies_clean")

print("\n>>> Tipo de dato de fechas (deben ser 'date'):")
policies_silver.select("POL_EFF_DATE", "POL_EXPIRY_DATE", "PREMIUM").printSchema()

print("\n>>> Nulos en campos clave:")
policies_silver.select(
    F.count(F.when(F.col("POLICY_NO").isNull(), 1)).alias("nulos_POLICY_NO"),
    F.count(F.when(F.col("CUST_ID").isNull(), 1)).alias("nulos_CUST_ID"),
    F.count(F.when(F.col("PREMIUM").isNull(), 1)).alias("nulos_PREMIUM"),
    F.count(F.when(F.col("POL_EFF_DATE").isNull(), 1)).alias("nulos_POL_EFF_DATE"),
    F.count(F.when(F.col("POL_EXPIRY_DATE").isNull(), 1)).alias("nulos_POL_EXPIRY_DATE"),
).show()

print("\n>>> Validación: filas donde fecha inicio > fecha fin (debe ser 0):")
policies_silver.filter(F.col("POL_EFF_DATE") > F.col("POL_EXPIRY_DATE")).count()

# ------------------------------------------------------------
# 3. CLAIMS CLEAN
# ------------------------------------------------------------
print("=" * 60)
print("CLAIMS CLEAN")
print("=" * 60)

claims_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.claims_clean")

print("\n>>> Tipo de dato de fechas (deben ser 'date'):")
claims_silver.select("claim_date", "license_issue_date").printSchema()

print("\n>>> Nulos en fechas y campos clave:")
claims_silver.select(
    F.count(F.when(F.col("claim_no").isNull(), 1)).alias("nulos_claim_no"),
    F.count(F.when(F.col("policy_no").isNull(), 1)).alias("nulos_policy_no"),
    F.count(F.when(F.col("claim_date").isNull(), 1)).alias("nulos_claim_date"),
    F.count(F.when(F.col("license_issue_date").isNull(), 1)).alias("nulos_license_issue_date"),
    F.count(F.when(F.col("total") < 0, 1)).alias("total_negativo"),
    F.count(F.when(F.col("age") <= 0, 1)).alias("age_invalida"),
).show()

print("\n>>> Muestra de fechas convertidas:")
claims_silver.select("claim_no", "claim_date", "license_issue_date").show(5)

# ------------------------------------------------------------
# 4. TELEMATICS CLEAN
# ------------------------------------------------------------
print("=" * 60)
print("TELEMATICS CLEAN")
print("=" * 60)

telematics_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.telematics_clean")

print("\n>>> Tipo de dato de event_timestamp (debe ser 'timestamp'):")
telematics_silver.select("event_timestamp").printSchema()

print("\n>>> Nulos y coordenadas inválidas (todos deben ser 0):")
telematics_silver.select(
    F.count(F.when(F.col("chassis_no").isNull(), 1)).alias("nulos_chassis_no"),
    F.count(F.when(F.col("event_timestamp").isNull(), 1)).alias("nulos_timestamp"),
    F.count(F.when(F.col("latitude").isNull(), 1)).alias("nulos_latitude"),
    F.count(F.when(F.col("longitude").isNull(), 1)).alias("nulos_longitude"),
    F.count(F.when(~F.col("latitude").between(-90, 90), 1)).alias("latitud_invalida"),
    F.count(F.when(~F.col("longitude").between(-180, 180), 1)).alias("longitud_invalida"),
    F.count(F.when(F.col("speed") < 0, 1)).alias("velocidad_negativa"),
).show()

print("\n>>> Rango de coordenadas:")
telematics_silver.select(
    F.min("latitude").alias("lat_min"),
    F.max("latitude").alias("lat_max"),
    F.min("longitude").alias("lon_min"),
    F.max("longitude").alias("lon_max"),
    F.min("speed").alias("speed_min"),
    F.max("speed").alias("speed_max"),
).show()

# ------------------------------------------------------------
# 5. TRAINING IMAGES
# ------------------------------------------------------------
print("=" * 60)
print("TRAINING IMAGES")
print("=" * 60)

training_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.training_images")

print("\n>>> Nulos en campos clave (deben ser 0):")
training_silver.select(
    F.count(F.when(F.col("path").isNull(), 1)).alias("nulos_path"),
    F.count(F.when(F.col("image_name").isNull() | (F.col("image_name") == ""), 1)).alias("nulos_image_name"),
    F.count(F.when(F.col("label").isNull() | (F.col("label") == ""), 1)).alias("nulos_label"),
).show()

print("\n>>> Labels distintos encontrados:")
training_silver.groupBy("label").count().orderBy("label").show()

print("\n>>> Muestra:")
training_silver.select("path", "image_name", "label").show(5, truncate=False)

# ------------------------------------------------------------
# 6. CLAIM IMAGES
# ------------------------------------------------------------
print("=" * 60)
print("CLAIM IMAGES")
print("=" * 60)

claim_images_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.claim_images")

print("\n>>> Nulos en campos clave (deben ser 0):")
claim_images_silver.select(
    F.count(F.when(F.col("path").isNull(), 1)).alias("nulos_path"),
    F.count(F.when(F.col("image_name").isNull() | (F.col("image_name") == ""), 1)).alias("nulos_image_name"),
).show()

print("\n>>> Muestra:")
claim_images_silver.select("path", "image_name").show(5, truncate=False)

# ------------------------------------------------------------
# 7. CLAIM IMAGES METADATA CLEAN
# ------------------------------------------------------------
print("=" * 60)
print("CLAIM IMAGES METADATA CLEAN")
print("=" * 60)

metadata_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.claim_images_metadata_clean")

print("\n>>> Nulos en campos clave (deben ser 0):")
metadata_silver.select(
    F.count(F.when(F.col("image_name").isNull(), 1)).alias("nulos_image_name"),
    F.count(F.when(F.col("claim_no").isNull(), 1)).alias("nulos_claim_no"),
    F.count(F.when(F.col("chassis_no").isNull(), 1)).alias("nulos_chassis_no"),
).show()

print("\n>>> Duplicados por image_name (debe ser 0):")
duplicados = (
    metadata_silver
    .groupBy("image_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Filas duplicadas por image_name: {duplicados}")

print("\n>>> Validación de join con claim_images: cuántas imágenes tienen metadata:")
joined = claim_images_silver.join(metadata_silver, on="image_name", how="inner")
print(f"Imágenes con metadata disponible: {joined.count()}")
print(f"Total claim images: {claim_images_silver.count()}")

print("\n>>> Muestra de metadata:")
metadata_silver.show(5, truncate=False)

# ------------------------------------------------------------
# RESUMEN FINAL
# ------------------------------------------------------------
print("=" * 60)
print("RESUMEN FINAL - CONTEO DE FILAS POR TABLA")
print("=" * 60)

tablas = {
    "customers_clean":            customers_silver,
    "policies_clean":             policies_silver,
    "claims_clean":               claims_silver,
    "telematics_clean":           telematics_silver,
    "training_images":            training_silver,
    "claim_images":               claim_images_silver,
    "claim_images_metadata_clean":metadata_silver,
}

for nombre, df in tablas.items():
    print(f"  {nombre}: {df.count():,} filas")